# TOML - Python

All 7 Python examples from [docs/toml.md](https://platob.github.io/yggdryl/toml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import toml

source = 'title = "yggdryl"\ncount = 3\n\n[owner]\nname = "Ada"\n'
value = toml.loads(source)

assert value == {"title": "yggdryl", "count": 3, "owner": {"name": "Ada"}}
assert value["owner"]["name"] == "Ada"
assert toml.loads(toml.dumps(value)) == value

## Table order

In [ ]:
from yggdryl import toml

value = {"zeta": 1, "beta": 2, "alpha": {"deep": 3}}

encoded = toml.dumps(value)
assert encoded == b'"zeta" = 1\n"beta" = 2\n"alpha" = {"deep" = 3}\n'
assert list(toml.loads(encoded)) == ["zeta", "beta", "alpha"]

## The type mapping

In [ ]:
import datetime as dt
import math

from yggdryl import toml

value = toml.loads(
    'text = "café"\n'
    "integer = 7\n"
    "hex = 0x2a\n"
    "float = 1.5\n"
    "infinite = inf\n"
    "negative_zero = -0.0\n"
    "flag = true\n"
    'array = [1, "two"]\n'
    "table = { nested = 1 }\n"
    "moment = 1979-05-27T07:32:00Z\n"
)

assert isinstance(value["text"], str)
assert isinstance(value["integer"], int) and not isinstance(value["integer"], bool)
assert value["hex"] == 42
assert isinstance(value["float"], float)
assert value["infinite"] == math.inf
assert math.copysign(1.0, value["negative_zero"]) == -1.0
assert value["flag"] is True
assert value["array"] == [1, "two"]
assert value["table"] == {"nested": 1}
assert value["moment"] == dt.datetime(1979, 5, 27, 7, 32, tzinfo=dt.timezone.utc)

## Values TOML has no syntax for

In [ ]:
from decimal import Decimal

from yggdryl import toml

value = {"missing": None, "blob": b"\x00\xff", "price": Decimal("1.25")}

encoded = toml.dumps(value)
assert b'"missing" = { "$yggdryl" = { version = 1, type = "null" } }' in encoded
assert toml.loads(encoded) == value

# A TOML root is a table, so a non-table root is wrapped the same way.
assert toml.loads(toml.dumps("scalar root")) == "scalar root"

# A user table that only looks like an envelope stays user data.
lookalike = {"$yggdryl": {"version": 1, "type": "null"}}
assert toml.loads(toml.dumps(lookalike)) == lookalike

## Dates and times

In [ ]:
import datetime as dt

from yggdryl import toml

value = toml.loads(
    "offset = 1979-05-27T07:32:00Z\n"
    "local = 1979-05-27T07:32:00\n"
    "day = 1979-05-27\n"
    "clock = 07:32:00\n"
)

assert value["offset"] == dt.datetime(1979, 5, 27, 7, 32, tzinfo=dt.timezone.utc)
assert value["local"] == dt.datetime(1979, 5, 27, 7, 32)
assert value["day"] == dt.date(1979, 5, 27)
assert value["clock"] == dt.time(7, 32)

# Each form goes back out in the syntax it arrived in.
assert b'"day" = 1979-05-27\n' in toml.dumps(value)

## Exactly one document

In [ ]:
from yggdryl import toml

# The root is a table, so an empty or comment-only document is an empty table.
assert toml.loads("# nothing to see\n") == {}
assert toml.dumps({}) == b""

# There is no multi-document pair, only the single-document one.
assert toml.load is toml.loads
assert not hasattr(toml, "loads_all")
assert not hasattr(toml, "dumps_all")

## Failures

In [ ]:
from yggdryl import toml

try:
    toml.loads("ok = 0\nnested = { a = 1, a = 2 }\n")
except ValueError as error:
    assert "toml" in str(error)
    assert "duplicate" in str(error)
else:
    raise AssertionError("duplicate keys must be rejected")

try:
    toml.loads("big = 9223372036854775808")
except ValueError:
    pass
else:
    raise AssertionError("an out-of-range integer must be rejected")

# Depth is checked before anything is written.
deep = None
for index in range(32):
    deep = {index: deep}
try:
    toml.dumps(deep)
except ValueError as error:
    assert "hard limit" in str(error)
else:
    raise AssertionError("an over-deep value must be rejected")